# Run CROCUS Flows on GCE

## Imports

In [19]:
import time
import json
import datetime
import globus_sdk

from globus_sdk import TimerJob
from globus_compute_sdk import Executor
from globus_sdk.experimental.globus_app import UserApp

from globus_sdk.utils import slash_join

In [20]:
CLIENT_ID = "c781864e-a9c9-482e-8db8-d58ac5962a86"
my_app = UserApp("crocus-user-app", client_id=CLIENT_ID)

flows_client = globus_sdk.FlowsClient(app=my_app)

In [21]:
compute_endpoint = "28700b55-71b8-485f-b126-df7366462a6e"
wxt_function = "502fb462-dca3-44af-800b-74eeb347103f"
aqt_function = "de1d00ef-deb3-446f-b1a3-8ab75b3b4be4"

In [22]:
gce = Executor(endpoint_id=compute_endpoint)

In [26]:
# Prepare payload for ESGF ingest-wxt
wxt_data = {
    "ndays": 1,
    "y": 2025,
    "m": 3,
    "d": 13,
    "site": 'NEIU',
    "hours": 24,
    "odir": "/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/neiu/neiu-wxt-a1"
}

# Start the task
future = gce.submit_to_registered_function(wxt_function, kwargs=wxt_data)

In [6]:
# Wait and print the result
result = future.result()
print(result)

['/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/neiu/neiu-wxt-a1/crocus-NEIU-wxt-a1_20250313_000000.nc']


/Users/mgrover/mambaforge/envs/esgf-crocus/lib/python3.10/site-packages/globus_compute_sdk/sdk/client.py:269: UserWarning: 
Environment differences detected between local SDK and endpoint 28700b55-71b8-485f-b126-df7366462a6e workers:
	    SDK: Python 3.10.16/Dill 0.3.5.1
	Workers: Python 3.11.11/Dill 0.3.9
This may cause serialization issues.  See https://globus-compute.readthedocs.io/en/latest/sdk.html#avoiding-serialization-errors for more information.
  warnings.warn(check_result, UserWarning)


In [27]:
# Prepare payload for ESGF ingest-aqt
aqt_data = {
    "ndays": 1,
    "y": 2024,
    "m": 8,
    "d": 2,
    "site": 'NEIU',
    "hours": 1,
    "odir": "/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/neiu/neiu-aqt-a1"
}

# Start the task
future = gce.submit_to_registered_function(aqt_function, kwargs=aqt_data)

# Wait and print the result
result = future.result()
print(result)

KeyboardInterrupt: 

In [44]:
flow_definition = {
    "Comment": "A Node-Level Processing Workflow for WXT and AQT",
    "StartAt": "ProcessWXT",
    "States": {
        "ProcessWXT": {
            "Comment": "Collect WXT data from Sage",
            "Type": "Action",
            "ActionUrl": "https://compute.actions.globus.org/",
            "Parameters": {
                "endpoint.$": "$.input.compute_endpoint",
                "function.$": "$.input.wxt_function",
                "kwargs.$": "$.input.wxt_kwargs"
            },
            "ResultPath": "$.WXT_output",
            "WaitTime": 3000,
            "Next": "ProcessAQT"
        },
        "ProcessAQT": {
            "Comment": "Collect WXT data from Sage",
            "Type": "Action",
            "ActionUrl": "https://compute.actions.globus.org/",
            "Parameters": {
                "endpoint.$": "$.input.compute_endpoint",
                "function.$": "$.input.aqt_function",
                "kwargs.$": "$.input.aqt_kwargs"
            },
            "ResultPath": "$.AQT_output",
            "WaitTime": 1200,
            "End": True
        },
    }
}

In [45]:
flow_input = {
    "input": {
        "compute_endpoint": compute_endpoint,
        "wxt_kwargs": wxt_data,
        "wxt_function": wxt_function,
        "aqt_kwargs": aqt_data,
        "aqt_function": aqt_function,
    }
}

In [46]:
flow = flows_client.create_flow(title="CROCUS Flow on GCE", definition=flow_definition, input_schema={})

In [47]:
flow_id = flow['id']
flow_id

'dcaf309c-0271-4b2d-be5e-b60519020651'

In [48]:
specific_flow_client = globus_sdk.SpecificFlowClient(
    flow_id=flow_id,
    app=my_app,
)

In [49]:
run = specific_flow_client.run_flow(
  body=flow_input,
  label="CROCUS GCE Example",
  tags=['CROCUS', 'example', 'gce']
)


Please authenticate with Globus here:
-------------------------------------
https://auth.globus.org/v2/oauth2/authorize?client_id=c781864e-a9c9-482e-8db8-d58ac5962a86&redirect_uri=https%3A%2F%2Fauth.globus.org%2Fv2%2Fweb%2Fauth-code&scope=openid+https%3A%2F%2Fauth.globus.org%2Fscopes%2Feec9b274-0c81-4334-bdc2-54e90e689b9a%2Fall+https%3A%2F%2Fauth.globus.org%2Fscopes%2Fdcaf309c-0271-4b2d-be5e-b60519020651%2Fflow_dcaf309c_0271_4b2d_be5e_b60519020651_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F50b37a25-28ad-4dda-8a2e-6960a6c5e856%2Fflow_50b37a25_28ad_4dda_8a2e_6960a6c5e856_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F524230d7-ea86-4a52-8312-86065a9e0417%2Ftimer%5Bhttps%3A%2F%2Fauth.globus.org%2Fscopes%2F50b37a25-28ad-4dda-8a2e-6960a6c5e856%2Fflow_50b37a25_28ad_4dda_8a2e_6960a6c5e856_user%5D&state=_default&response_type=code&code_challenge=tHBbw1tKRSv03VmS_3-ZjXr41BYw8Y5KQdDpjKvYqwE&code_challenge_method=S256&access_type=online&prefill_named_grant=crocus-user-app+on+evswl145.evs.anl.go

Enter the resulting Authorization Code here:  XiGOZOJ3LcYWB5mKyTXbPACmn3ZPyy


In [119]:
# Get run details
# run = flows_client.get_run(run_id)

run_id = run['run_id']
run_status = run['status']
print("This flow can be monitored in the Web App:")
print(f"https://app.globus.org/runs/{run_id}")
print(f"Flow run started with ID: {run_id} - Status: {run_status}")

# Poll the Flow ser/vice to check on the status of the flow
while run_status == 'ACTIVE':
    time.sleep(5)
    run = flows_client.get_run(run_id)
    run_status = run['status']
    print(f'Run status: {run_status}')
    
# Run completed
print(json.dumps(run.data, indent=2))

This flow can be monitored in the Web App:
https://app.globus.org/runs/66e1b633-36cd-41e4-a780-f5ca6977a5ad
Flow run started with ID: 66e1b633-36cd-41e4-a780-f5ca6977a5ad - Status: ACTIVE


KeyboardInterrupt: 

In [50]:
input_schema = {
    "required": [
        "input"
    ],
    "properties": {
        "input": {
            "type": "object",
            "required": [
                "compute_endpoint",
                "wxt_function",
                "wxt_kwargs",
                "aqt_function",
                "aqt_kwargs"
            ],
            "properties": {
                "compute_endpoint": {
                    "type": "string",
                    "format": "uuid",
                    "default": compute_endpoint,
                    "title": "Globus Compute Endpoint ID",
                    "description": "The UUID of the Globus Compute endpoint where the function will run"
                },
                "wxt_function": {
                    "type": "string",
                    "format": "uuid",
                    "default": wxt_function,
                    "title": "Globus Compute Function ID",
                    "description": "The UUID of the function to invoke; must be registered with the Globus Compute service"
                },
                "wxt_kwargs": {
                    "type": "object",
                    "title": "Function Inputs",
                    "description": "Inputs to pass to the function",
                    "properties":  {
                        "ndays": {
                            "type": "integer",
                            "default": 1
                        },
                        "y": {
                            "type": "integer",
                            "default": 2024
                        },
                        "m": {
                            "type": "integer",
                            "default": 8
                        },
                        "d": {
                            "type": "integer",
                            "default": 1
                        },
                        "site": {
                            "type": "string",
                            "default": "NU"
                        },
                        "hours": {
                            "type": "integer",
                            "default": 24
                        },
                        "odir": {
                            "type": "string",
                            "default": "/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/nu/nu-aqt-a1"
                        },
                    },
                    "additionalProperties": False
                },
                "aqt_function": {
                    "type": "string",
                    "format": "uuid",
                    "default": aqt_function,
                    "title": "Globus Compute Function ID",
                    "description": "The UUID of the function to invoke; must be registered with the Globus Compute service"
                },
                "aqt_kwargs": {
                    "type": "object",
                    "title": "Function Inputs",
                    "description": "Inputs to pass to the function",
                    "properties":  {
                        "ndays": {
                            "type": "integer",
                            "default": 1
                        },
                        "y": {
                            "type": "integer",
                            "default": 2024
                        },
                        "m": {
                            "type": "integer",
                            "default": 8
                        },
                        "d": {
                            "type": "integer",
                            "default": 1
                        },
                        "site": {
                            "type": "string",
                            "default": "NU"
                        },
                        "hours": {
                            "type": "integer",
                            "default": 1
                        },
                        "odir": {
                            "type": "string",
                            "default": "/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/nu/nu-aqt-a1"
                        },
                    },
                    "additionalProperties": False
                }
            },
            "additionalProperties": False
        }
    },
    "additionalProperties": False
}

In [51]:
flow = flows_client.update_flow(flow_id=flow_id, title="CROCUS GCE Flow", definition=flow_definition, input_schema=input_schema)

In [52]:
from globus_sdk.scopes import TimerScopes, FlowsScopes
from globus_sdk import TimerClient

In [53]:
flow_scope = specific_flow_client.scopes.user
end_scope = f"{TimerScopes.timer}[{flow_scope}]"

timer_client = TimerClient(app=my_app, app_scopes=end_scope)

In [54]:
callback_url = slash_join(specific_flow_client.base_url, f"/flows/{flow_id}/run")

In [55]:
def build_timer_input_for_site(site):
    timer_input = {
        'input': {
            'compute_endpoint': compute_endpoint,
            'wxt_kwargs': {
                'ndays': 1,
                'site': f'{site.upper()}',
                'hours': 24,
                'odir': f"/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/{site.lower()}/{site.lower()}-wxt-a1"
            },
            'wxt_function': wxt_function,
            'aqt_kwargs': {
                'ndays': 1,
                'site': f'{site.upper()}',
                'hours': 24,
                'odir': f'/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/{site.lower()}/{site.lower()}-aqt-a1'
            },
            'aqt_function': aqt_function,
        }
    }
    return timer_input

In [56]:
sites = ['NU',
         'CSU',
         'NEIU',
         'ATMOS',
         'UIC',
         'NEIU_CCICS',
         "BIG",
         'HUM',
         "DOWN",
         "SHEDD"]

In [57]:
callback_url = slash_join(specific_flow_client.base_url, f"/flows/{flow_id}/run")
flow_input = build_timer_input_for_site(site)
timer = TimerJob(
    callback_url=callback_url,
    callback_body={"body": flow_input, "label": f"CROCUS-Flows GCE Timer Flow {site}"},
    start=datetime.datetime.utcnow(),
    interval=datetime.timedelta(seconds=3600),
    scope=flow_scope,
    name=f"CROCUS-Flows GCE Timer {site}",
)
response = timer_client.create_job(timer)
job_id = timer_client.get_job(response.get('job_id')).data
job_id


Please authenticate with Globus here:
-------------------------------------
https://auth.globus.org/v2/oauth2/authorize?client_id=c781864e-a9c9-482e-8db8-d58ac5962a86&redirect_uri=https%3A%2F%2Fauth.globus.org%2Fv2%2Fweb%2Fauth-code&scope=openid+https%3A%2F%2Fauth.globus.org%2Fscopes%2Feec9b274-0c81-4334-bdc2-54e90e689b9a%2Fall+https%3A%2F%2Fauth.globus.org%2Fscopes%2Fdcaf309c-0271-4b2d-be5e-b60519020651%2Fflow_dcaf309c_0271_4b2d_be5e_b60519020651_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F50b37a25-28ad-4dda-8a2e-6960a6c5e856%2Fflow_50b37a25_28ad_4dda_8a2e_6960a6c5e856_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F524230d7-ea86-4a52-8312-86065a9e0417%2Ftimer%5Bhttps%3A%2F%2Fauth.globus.org%2Fscopes%2Fdcaf309c-0271-4b2d-be5e-b60519020651%2Fflow_dcaf309c_0271_4b2d_be5e_b60519020651_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F50b37a25-28ad-4dda-8a2e-6960a6c5e856%2Fflow_50b37a25_28ad_4dda_8a2e_6960a6c5e856_user%5D&state=_default&response_type=code&code_challenge=DWaqQTaHNPBzSp-k8xNLj


KeyboardInterrupt



In [43]:
for job in timer_client.list_jobs().data["jobs"]:
    timer_client.delete_job(job["job_id"])

In [58]:
jobs = []
for site in sites[1:]:
    callback_url = slash_join(specific_flow_client.base_url, f"/flows/{flow_id}/run")
    flow_input = build_timer_input_for_site(site)
    timer = TimerJob(
        callback_url=callback_url,
        callback_body={"body": flow_input, "label": f"CROCUS-Flows GCE Timer Flow {site}"},
        start=datetime.datetime.utcnow(),
        interval=datetime.timedelta(seconds=3600),
        scope=flow_scope,
        name=f"CROCUS-Flows GCE Timer {site}",
    )
    response = timer_client.create_job(timer)
    job_id = timer_client.get_job(response.get('job_id')).data
    print(f"Submitted flow for {site}")
    print(job_id)
    jobs.append(job_id)
    time.sleep(120)


Please authenticate with Globus here:
-------------------------------------
https://auth.globus.org/v2/oauth2/authorize?client_id=c781864e-a9c9-482e-8db8-d58ac5962a86&redirect_uri=https%3A%2F%2Fauth.globus.org%2Fv2%2Fweb%2Fauth-code&scope=openid+https%3A%2F%2Fauth.globus.org%2Fscopes%2Feec9b274-0c81-4334-bdc2-54e90e689b9a%2Fall+https%3A%2F%2Fauth.globus.org%2Fscopes%2Fdcaf309c-0271-4b2d-be5e-b60519020651%2Fflow_dcaf309c_0271_4b2d_be5e_b60519020651_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F50b37a25-28ad-4dda-8a2e-6960a6c5e856%2Fflow_50b37a25_28ad_4dda_8a2e_6960a6c5e856_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F524230d7-ea86-4a52-8312-86065a9e0417%2Ftimer%5Bhttps%3A%2F%2Fauth.globus.org%2Fscopes%2Fdcaf309c-0271-4b2d-be5e-b60519020651%2Fflow_dcaf309c_0271_4b2d_be5e_b60519020651_user+https%3A%2F%2Fauth.globus.org%2Fscopes%2F50b37a25-28ad-4dda-8a2e-6960a6c5e856%2Fflow_50b37a25_28ad_4dda_8a2e_6960a6c5e856_user%5D&state=_default&response_type=code&code_challenge=5V9qzJORR_JdReZWAdoz9

Enter the resulting Authorization Code here:  E4uhvM6eLEjW2kWnHbnfzK0tWbxJnZ


Submitted flow for CSU
{'name': 'CROCUS-Flows GCE Timer CSU', 'stop_after': None, 'interval': 3600.0, 'scope': 'https://auth.globus.org/scopes/dcaf309c-0271-4b2d-be5e-b60519020651/flow_dcaf309c_0271_4b2d_be5e_b60519020651_user', 'callback_url': 'https://flows.automate.globus.org/flows/dcaf309c-0271-4b2d-be5e-b60519020651/run', 'callback_body': {'body': {'input': {'compute_endpoint': '28700b55-71b8-485f-b126-df7366462a6e', 'wxt_kwargs': {'ndays': 1, 'site': 'CSU', 'hours': 24, 'odir': '/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/csu/csu-wxt-a1'}, 'wxt_function': '502fb462-dca3-44af-800b-74eeb347103f', 'aqt_kwargs': {'ndays': 1, 'site': 'CSU', 'hours': 24, 'odir': '/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/csu/csu-aqt-a1'}, 'aqt_function': 'de1d00ef-deb3-446f-b1a3-8ab75b3b4be4'}}, 'label': 'CROCUS-Flows GCE Timer Flow CSU'}, 'start': '2025-03-18T17:31:15+00:00', 'inactive_reason': None, 'job_id': '84cba5ba-7c0d-4e7a-b78c-b73861cf51bd', 'status': 'loaded'

/Users/mgrover/mambaforge/envs/esgf-crocus/lib/python3.10/site-packages/globus_compute_sdk/sdk/client.py:269: UserWarning: 
Environment differences detected between local SDK and endpoint 28700b55-71b8-485f-b126-df7366462a6e workers:
	    SDK: Python 3.10.16/Dill 0.3.5.1
	Workers: Python 3.11.11/Dill 0.3.9
This may cause serialization issues.  See https://globus-compute.readthedocs.io/en/latest/sdk.html#avoiding-serialization-errors for more information.
  warnings.warn(check_result, UserWarning)


Submitted flow for HUM
{'name': 'CROCUS-Flows GCE Timer HUM', 'stop_after': None, 'interval': 3600.0, 'scope': 'https://auth.globus.org/scopes/dcaf309c-0271-4b2d-be5e-b60519020651/flow_dcaf309c_0271_4b2d_be5e_b60519020651_user', 'callback_url': 'https://flows.automate.globus.org/flows/dcaf309c-0271-4b2d-be5e-b60519020651/run', 'callback_body': {'body': {'input': {'compute_endpoint': '28700b55-71b8-485f-b126-df7366462a6e', 'wxt_kwargs': {'ndays': 1, 'site': 'HUM', 'hours': 24, 'odir': '/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/hum/hum-wxt-a1'}, 'wxt_function': '502fb462-dca3-44af-800b-74eeb347103f', 'aqt_kwargs': {'ndays': 1, 'site': 'HUM', 'hours': 24, 'odir': '/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/hum/hum-aqt-a1'}, 'aqt_function': 'de1d00ef-deb3-446f-b1a3-8ab75b3b4be4'}}, 'label': 'CROCUS-Flows GCE Timer Flow HUM'}, 'start': '2025-03-18T17:43:30+00:00', 'inactive_reason': None, 'job_id': '8b8c3383-afe7-42f0-a889-2de34766a04a', 'status': 'loaded'

Channel closed.  Reopening.
  <Channel number=1 CLOSED conn=<SelectConnection CLOSED transport=None params=<URLParameters host=compute.amqps.globus.org port=443 virtual_host=/ ssl=True>>>
  (No activity or too many missed heartbeats in the last 60 seconds)
_ResultWatcher<✗; pid=2393; tg=a6779a96-d936-4cc8-baa5-a9c650324298; fut=2; res=0; qp=tg_result.85e67bd6-4b28-46e3-a2fd-3d046c6dafcf:> Unhandled error; shutting down
Traceback (most recent call last):
  File "/Users/mgrover/mambaforge/envs/esgf-crocus/lib/python3.10/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
  File "/Users/mgrover/mambaforge/envs/esgf-crocus/lib/python3.10/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
  File "/Users/mgrover/mambaforge/envs/esgf-crocus/lib/python3.10/socket.py", line 967, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, ty

Submitted flow for SHEDD
{'name': 'CROCUS-Flows GCE Timer SHEDD', 'stop_after': None, 'interval': 3600.0, 'scope': 'https://auth.globus.org/scopes/dcaf309c-0271-4b2d-be5e-b60519020651/flow_dcaf309c_0271_4b2d_be5e_b60519020651_user', 'callback_url': 'https://flows.automate.globus.org/flows/dcaf309c-0271-4b2d-be5e-b60519020651/run', 'callback_body': {'body': {'input': {'compute_endpoint': '28700b55-71b8-485f-b126-df7366462a6e', 'wxt_kwargs': {'ndays': 1, 'site': 'SHEDD', 'hours': 24, 'odir': '/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/shedd/shedd-wxt-a1'}, 'wxt_function': '502fb462-dca3-44af-800b-74eeb347103f', 'aqt_kwargs': {'ndays': 1, 'site': 'SHEDD', 'hours': 24, 'odir': '/nfs/gce/projects/crocus/data/ingested-data/long-term-sites/shedd/shedd-aqt-a1'}, 'aqt_function': 'de1d00ef-deb3-446f-b1a3-8ab75b3b4be4'}}, 'label': 'CROCUS-Flows GCE Timer Flow SHEDD'}, 'start': '2025-03-18T17:56:44+00:00', 'inactive_reason': None, 'job_id': '90160935-0f47-443e-9670-cecb4223290d', 

Channel closed.  Reopening.
  <Channel number=1 CLOSED conn=<SelectConnection CLOSED transport=None params=<URLParameters host=compute.amqps.globus.org port=443 virtual_host=/ ssl=True>>>
  (Stream connection lost: SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2578)'))
